In [1]:
"""
Module A: Dynamic Outlier Detection & Clean Dataset Export
==========================================================
"""

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

# -----------------------------------------------------------------
# 1. LOAD ORIGINAL RAW DATA
# -----------------------------------------------------------------
INPUT_CSV = "isro_burnin_ess_synthetic_dataset_cleaned.csv"

# Load original raw file directly without dropping or renaming any columns
raw_df = pd.read_csv(INPUT_CSV)

# Detect data working frame if metadata lines exist
work_df = raw_df.copy()
if "component_id" not in work_df.columns and "lot_id" not in work_df.columns:
    work_df = pd.read_csv(INPUT_CSV, skiprows=1)

param_mapping = {
    "iddq_uA": "iddq",
    "leakage_current_nA": "leakage",
    "propagation_delay_ns": "prop_delay",
}

melted_list = []
for prefix, param_name in param_mapping.items():
    sub = pd.DataFrame(
        {
            "lot_id": work_df["lot_id"],
            "part_id": work_df["component_id"],
            "param_name": param_name,
            "value_0h": work_df[f"{prefix}_0h"],
            "value_24h": work_df[f"{prefix}_24h"],
            "value_96h": work_df[f"{prefix}_96h"],
            "value_168h": work_df[f"{prefix}_168h"],
        }
    )
    melted_list.append(sub)

df = pd.concat(melted_list, ignore_index=True)



In [2]:

# -----------------------------------------------------------------
# 2. FEATURE ENGINEERING (SAFE DRIFT RATES)
# -----------------------------------------------------------------
df["drift_0_24"] = (df["value_24h"] - df["value_0h"]) / 24.0
df["drift_24_96"] = (df["value_96h"] - df["value_24h"]) / 72.0
df["drift_96_168"] = (df["value_168h"] - df["value_96h"]) / 72.0

df["total_drift_pct"] = np.where(
    df["value_0h"] != 0,
    (df["value_168h"] - df["value_0h"]) / df["value_0h"],
    0.0,
)


In [3]:


# -----------------------------------------------------------------
# 3. ROBUST DPAT (Median + MAD, per lot, per parameter)
# -----------------------------------------------------------------
def robust_dpat_limits(series, k=6):
    median = np.median(series)
    mad = np.median(np.abs(series - median))
    sigma_robust = 1.4826 * mad
    upl = median + k * sigma_robust
    lpl = median - k * sigma_robust
    return median, sigma_robust, upl, lpl


def apply_robust_dpat(df, value_col, group_cols=("lot_id", "param_name"), k=6):
    out_rows = []
    for keys, group in df.groupby(list(group_cols)):
        median, sigma, upl, lpl = robust_dpat_limits(
            group[value_col].values, k=k
        )
        g = group.copy()
        g[f"{value_col}_median"] = median
        g[f"{value_col}_sigma_robust"] = sigma
        g[f"{value_col}_UPL"] = upl
        g[f"{value_col}_LPL"] = lpl
        g[f"{value_col}_DPAT_outlier"] = (g[value_col] > upl) | (
            g[value_col] < lpl
        )
        out_rows.append(g)
    return pd.concat(out_rows, ignore_index=True)


df = apply_robust_dpat(df, "value_0h", k=6)
df = apply_robust_dpat(df, "total_drift_pct", k=6)



In [4]:
# -----------------------------------------------------------------
# 4. ISOLATION FOREST (per parameter, multivariate)
# -----------------------------------------------------------------
iso_results = []
feature_cols = [
    "value_0h",
    "value_24h",
    "value_96h",
    "value_168h",
    "drift_0_24",
    "drift_24_96",
    "drift_96_168",
    "total_drift_pct",
]

for param, group in df.groupby("param_name"):
    X = group[feature_cols].fillna(0)
    iso = IsolationForest(
        n_estimators=200, contamination="auto", random_state=42
    )

    g = group.copy()
    g["iso_forest_flag"] = iso.fit_predict(X) == -1
    g["iso_forest_score"] = iso.decision_function(X)
    iso_results.append(g)

df = pd.concat(iso_results, ignore_index=True)

In [5]:


# -----------------------------------------------------------------
# 5. COMBINE INTO FINAL VERDICT
# -----------------------------------------------------------------
df["final_flag"] = (
    df["value_0h_DPAT_outlier"]
    | df["total_drift_pct_DPAT_outlier"]
    | df["iso_forest_flag"]
)

df["flag_reason"] = df.apply(
    lambda r: ", ".join(
        filter(
            None,
            [
                "DPAT_0h" if r["value_0h_DPAT_outlier"] else None,
                "DPAT_drift" if r["total_drift_pct_DPAT_outlier"] else None,
                "IsolationForest" if r["iso_forest_flag"] else None,
            ],
        )
    ),
    axis=1,
)


In [6]:


# -----------------------------------------------------------------
# 6. ENRICHED EXPLANATORY OUTPUT REPORT
# -----------------------------------------------------------------
total_evals = len(df)
total_flagged_evals = df["final_flag"].sum()
rejected_part_ids = df[df["final_flag"]]["part_id"].unique()
unique_parts_count = len(rejected_part_ids)

dpat_0h_cnt = df["value_0h_DPAT_outlier"].sum()
dpat_drift_cnt = df["total_drift_pct_DPAT_outlier"].sum()
iso_cnt = df["iso_forest_flag"].sum()
total_components = work_df["component_id"].nunique()

print("=" * 70)
print("             MODULE A: DETAILED SCREENING REPORT")
print("=" * 70)
print(f"1. TOTAL EVALUATIONS CHECKED: {total_evals}")
print(
    f"   --> WHY: Raw dataset contains {total_components} IC components unpivoted across 3 parameters."
)
print(
    f"            ({total_components} parts * 3 params = {total_evals} test rows)"
)
print()
print(f"2. TOTAL PARAMETER FAILURES FLAGGED: {total_flagged_evals}")
print("   --> BREAKDOWN BY ANOMALY DETECTOR:")
print(f"       * Baseline DPAT Outliers (0h)    : {dpat_0h_cnt} evaluation(s)")
print(f"       * Burn-in Drift DPAT Outliers    : {dpat_drift_cnt} evaluation(s)")
print(f"       * Isolation Forest Anomaly Flags : {iso_cnt} evaluation(s)")
print(
    "   --> WHY: An evaluation row is flagged if it fails ANY of the 3 checks above."
)
print()
print(f"3. UNIQUE PHYSICAL IC COMPONENTS REJECTED: {unique_parts_count}")
print(
    f"   --> WHY: {unique_parts_count} out of {total_components} physical chips failed screening."
)
print(
    "            If ANY parameter of an IC fails (e.g. IDDQ drifts even if leakage passes),"
)
print("            the component is rejected from the lot.")
print("=" * 70)



             MODULE A: DETAILED SCREENING REPORT
1. TOTAL EVALUATIONS CHECKED: 30000
   --> WHY: Raw dataset contains 10000 IC components unpivoted across 3 parameters.
            (10000 parts * 3 params = 30000 test rows)

2. TOTAL PARAMETER FAILURES FLAGGED: 2081
   --> BREAKDOWN BY ANOMALY DETECTOR:
       * Baseline DPAT Outliers (0h)    : 281 evaluation(s)
       * Burn-in Drift DPAT Outliers    : 1500 evaluation(s)
       * Isolation Forest Anomaly Flags : 1959 evaluation(s)
   --> WHY: An evaluation row is flagged if it fails ANY of the 3 checks above.

3. UNIQUE PHYSICAL IC COMPONENTS REJECTED: 948
   --> WHY: 948 out of 10000 physical chips failed screening.
            If ANY parameter of an IC fails (e.g. IDDQ drifts even if leakage passes),
            the component is rejected from the lot.


In [7]:

# -----------------------------------------------------------------
# 7. EXPORT EXTREMELY FAITHFUL CLEAN CSV (ORIGINAL TABLE HEADERS)
# -----------------------------------------------------------------
# Save evaluation log
df.to_csv("module_a_output.csv", index=False)

# Filter original raw dataframe using part_id matching to preserve exact original columns
id_col = (
    "component_id"
    if "component_id" in raw_df.columns
    else raw_df.columns[raw_df.isin(rejected_part_ids).any()].tolist()[0]
)

clean_raw_df = raw_df[~raw_df[id_col].isin(rejected_part_ids)].copy()
clean_raw_df.to_csv("clean_burnin_dataset.csv", index=False)

print("\nFiles successfully created:")
print(" -> module_a_output.csv (Evaluation audit log)")
print(
    " -> clean_burnin_dataset.csv (Exact original CSV structure without outliers)"
)


Files successfully created:
 -> module_a_output.csv (Evaluation audit log)
 -> clean_burnin_dataset.csv (Exact original CSV structure without outliers)
